# Using the Resume Feature in QMCPy

The `resume` parameter lets you restart a stopped integration from where it left off.

**Typical workflow**:
1. Run `solution, data = sc.integrate()` with a loose tolerance for a quick estimate.
2. Save `data` to disk with `data.save(path)` so the state is preserved.
3. Later (or in a new session), load it: `data = Data.load(path)`.
4. Tighten the tolerance with `sc.set_tolerance(abs_tol=tighter)`.
5. Call `solution, data = sc.integrate(resume=data)` — only the *additional* samples needed are generated.

This avoids re-computing any work already done.

In [1]:
from qmcpy import CubQMCLatticeG, Genz, Gaussian, Lattice
from qmcpy.util.data import Data
import os, warnings
warnings.filterwarnings('ignore', category=UserWarning)

## Step 1: Quick Estimate (Loose Tolerance)

Define a 3-D Gaussian-weighted Genz oscillatory integral and run `CubQMCLatticeG` with a loose
tolerance to get a fast initial estimate.

In [2]:
discrete_distrib = Lattice(dimension=3)
true_measure = Gaussian(discrete_distrib, mean=0, covariance=1)
integrand = Genz(discrete_distrib, kind_func='oscillatory', kind_coeff=1)

abs_tol_loose = 1e-3
rel_tol = 0
solver = CubQMCLatticeG(integrand, abs_tol=abs_tol_loose, rel_tol=rel_tol)
solution1, data1 = solver.integrate()

print(f"Loose-tolerance solution:  {solution1.item():.8f}")
print(f"  Samples used:  {int(data1.n_total):,}")
print(f"  Time:          {data1.time_integrate:.4f} s")
print(f"  Error bound:   {(data1.comb_bound_high.item() - data1.comb_bound_low.item()) / 2:.2e}")

Loose-tolerance solution:  -0.42893166
  Samples used:  1,024
  Time:          0.0004 s
  Error bound:   9.73e-04


## Step 2: Save the Integration State

`data1.save(path)` pickles the `Data` object to disk.
Pass `compress=True` to create a smaller gzip-compressed file (`.gz` suffix added automatically).

In [3]:
os.makedirs('output', exist_ok=True)
save_path = 'output/resume_example_data1.pkl'
data1.save(save_path)
print(f"Saved to {save_path}  ({os.path.getsize(save_path):,} bytes)")

# Compressed variant — the .gz suffix is added automatically
data1.save('output/resume_example_data1_compressed.pkl', compress=True)
gz_size = os.path.getsize('output/resume_example_data1_compressed.pkl.gz')
print(f"Compressed file:  {gz_size:,} bytes  ({100*(1 - gz_size/os.path.getsize(save_path)):.0f}% smaller)")

Saved to output/resume_example_data1.pkl  (63,181 bytes)
Compressed file:  32,762 bytes  (48% smaller)


## Step 3: Resume with a Tighter Tolerance

Load the saved state and call `integrate(resume=loaded_data)` after tightening the tolerance.
Only the *incremental* samples needed to meet the new tolerance are generated — the previous work is fully reused.

In [4]:
loaded_data = Data.load(save_path)   # load from disk (simulates a new session)

abs_tol_tight = 1e-5
solver.set_tolerance(abs_tol=abs_tol_tight)
solution2, data2 = solver.integrate(resume=loaded_data)

print(f"Resumed solution:  {solution2.item():.8f}")
print(f"  Total samples:   {int(data2.n_total):,}  (was {int(data1.n_total):,} after Step 1)")
print(f"  New samples:     {int(data2.n_total - data1.n_total):,}")
print(f"  Time this step:  {data2.time_integrate:.4f} s")
print(f"  Error bound:     {(data2.comb_bound_high.item() - data2.comb_bound_low.item()) / 2:.2e}")

Resumed solution:  -0.42893209
  Total samples:   32,768  (was 1,024 after Step 1)
  New samples:     31,744
  Time this step:  0.0039 s
  Error bound:     9.33e-06


## Step 4: Compare with a Fresh Run

For reference, solve from scratch with the same tight tolerance to see how much work was saved.

In [5]:
solver_fresh = CubQMCLatticeG(
    Genz(Lattice(dimension=3), kind_func='oscillatory', kind_coeff=1),
    abs_tol=abs_tol_tight, rel_tol=rel_tol,
)
solution3, data3 = solver_fresh.integrate()

print(f"Fresh-start solution:  {solution3.item():.8f}")
print(f"  Samples used:        {int(data3.n_total):,}")
print(f"  Time:                {data3.time_integrate:.4f} s")
print()
print(f"Solutions match:  {abs(solution2.item() - solution3.item()) < 2 * abs_tol_tight}")

Fresh-start solution:  -0.42893209
  Samples used:        32,768
  Time:                0.0039 s

Solutions match:  True


## Key Takeaways

| | Resume approach | Fresh-start approach |
|---|---|---|
| Step 1 | Loose run → quick answer | — |
| Step 2 | Save state to disk | — |
| Step 3 | Resume with tight tolerance | Run from scratch |
| Samples | Loose + incremental only | All samples fresh |

**Benefits of resume**:
- If the loose run already used significant samples, the resume avoids re-generating them.
- State can be checkpointed to disk between sessions.
- Works for `CubQMCLatticeG`, `CubQMCNetG`, `CubBayesLatticeG`, `CubBayesNetG`, `CubMCCLT`, `CubMCG`, and `CubMCCLTVec`.